# 🎨 Kaggle All-in-One AI Studio (2x Tesla T4 16GB)
### Chuẩn OpenAI REST API trực tiếp qua Cloudflare Public Tunnel:
- 👁️ **VLM**: Qwen 26B (4-bit, Dual-GPU)
- 🎙️ **STT**: Whisper-large-v3-turbo (**Bản FULL FP16** trên GPU 0)
- 🔊 **TTS**: Kokoro-82M (**Bản FULL FP16** trên GPU 0)
- 🖼️ **GenImage**: FLUX.1-schnell (4-bit NF4 trên GPU 1)
- 🎬 **GenVideo**: Wan2.1-1.3B (Text-to-Video trên GPU 1)
- 🌐 **Endpoint**: Xuất trực tiếp URL Public chuẩn OpenAI (`/v1/...`)

In [ ]:
# 1. Nạp biến môi trường tự động (Hỗ trợ .env, Kaggle Secrets, hoặc cấu hình chuẩn)
import os, json

# Thiết lập mặc định
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_TOKEN"] = "hf_" + "zTCysSCpYtoKHhsAsyBSpQQVMospAnyQdl"
os.environ["KAGGLE_USERNAME"] = "nguynxuncngde180528"
os.environ["KAGGLE_KEY"] = "KGAT_8cf30e03c2129179e5e0870f50b86773"

# Đọc file .env nếu có sẵn
for env_candidate in [".env", "/kaggle/working/Gen_Image-Video/kaggle/all-in-one/.env"]:
    if os.path.exists(env_candidate):
        try:
            with open(env_candidate, "r") as ef:
                for line in ef:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        clean_val = v.strip().strip("\"").strip("\x27")
                        os.environ[k.strip()] = clean_val
        except Exception:
            pass

# Nạp Kaggle UserSecrets nếu được cấp quyền
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"]:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
except Exception:
    pass

# Tạo cấu hình ~/.kaggle/kaggle.json để Kaggle API / CLI hoạt động thông suốt
try:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as kf:
        json.dump({
            "username": os.environ["KAGGLE_USERNAME"],
            "key": os.environ["KAGGLE_KEY"]
        }, kf)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

print(f"✅ Môi trường cấu hình hoàn tất! User: {os.environ.get('KAGGLE_USERNAME')}")


In [ ]:
# 2. Đồng bộ mã nguồn Studio AI (GitHub + Live Payload)
import os, base64, io, tarfile

repo_dir = "/kaggle/working/Gen_Image-Video"
target_dir = "/kaggle/working/Gen_Image-Video/kaggle/all-in-one"

if not os.path.exists(repo_dir):
    !git clone https://github.com/nxc1802/Gen_Image-Video.git
else:
    !cd {repo_dir} && git pull origin main

%cd {target_dir}

payload_data = "..."
raw_tar = base64.b64decode(payload_data)
with tarfile.open(fileobj=io.BytesIO(raw_tar), mode="r:gz") as tar:
    tar.extractall(".")

print("✅ Toàn bộ module Studio AI phiên bản 3.5 đã đồng bộ thành công!")

In [ ]:
# 3. Cài đặt các thư viện cần thiết
!pip install -q -U torchao
!pip install -q -r requirements.txt

In [ ]:
# 4. Khởi động toàn bộ Studio và xuất Cloudflare Public URL
!while true; do     python -u main.py;     CODE=$?;     if [ $CODE -eq 0 ] || [ $CODE -eq 99 ]; then         echo "Clean exit requested ($CODE). Stopping notebook.";         break;     fi;     echo "Server exited with code $CODE, restarting in 3s...";     sleep 3; done
